<a href="https://colab.research.google.com/github/springboardmentor12458j/LiveMeetingSummarize/blob/Monica/AudioToText.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install vosk


In [25]:
# Download the small English Vosk model (Fixed syntax error)
!wget [https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip](https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip)

# Unzip the model file. Use -o to overwrite existing files, preventing prompts.
!unzip -o vosk-model-small-en-us-0.15.zip


/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `wget [https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip](https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip)'
Archive:  vosk-model-small-en-us-0.15.zip
  inflating: vosk-model-small-en-us-0.15/am/final.mdl  
  inflating: vosk-model-small-en-us-0.15/graph/disambig_tid.int  
  inflating: vosk-model-small-en-us-0.15/graph/HCLr.fst  
  inflating: vosk-model-small-en-us-0.15/graph/Gr.fst  
  inflating: vosk-model-small-en-us-0.15/graph/phones/word_boundary.int  
  inflating: vosk-model-small-en-us-0.15/conf/model.conf  
  inflating: vosk-model-small-en-us-0.15/conf/mfcc.conf  
  inflating: vosk-model-small-en-us-0.15/ivector/splice.conf  
  inflating: vosk-model-small-en-us-0.15/ivector/final.dubm  
  inflating: vosk-model-small-en-us-0.15/ivector/global_cmvn.stats  
  inflating: vosk-model-small-en-us-0.15/ivector/final.ie  
  inflating: vosk-model-small-en-us-0.15/ivect

In [27]:
import argparse
import queue
import sys
import json
import sounddevice as sd
from vosk import Model, KaldiRecognizer
import wave

# Audio settings
SAMPLE_RATE = 16000
CHANNELS = 1
BLOCKSIZE = 4000

# This path now correctly points to the folder extracted in Step 2
MODEL_PATH = "vosk-model-small-en-us-0.15"


def main():
    print("Processing audio from file: sample.wav\n")

    # load model
    try:
        print("Loading model from:", MODEL_PATH)
        model = Model(MODEL_PATH)
    except Exception as e:
        print("Failed to load model. Check if the model path is correct:", e, file=sys.stderr)
        print("Make sure you downloaded and extracted the VOSK model into the correct folder.")
        return

    rec = KaldiRecognizer(model, SAMPLE_RATE)
    rec.SetWords(False)

    transcript = ""
    prev_partial = ""

    try:
        wf = wave.open('sample.wav', 'rb')
        if wf.getnchannels() != CHANNELS or wf.getsampwidth() != 2 or wf.getframerate() != SAMPLE_RATE:
            print("Audio file must be mono, 16-bit PCM, and 16000 Hz sample rate.", file=sys.stderr)
            return

        while True:
            data = wf.readframes(BLOCKSIZE)
            if len(data) == 0:
                break # End of file

            if rec.AcceptWaveform(data):
                r = json.loads(rec.Result())
                text = r.get('text', '').strip()
                if text:
                    # append final text
                    if transcript:
                        transcript = transcript + " " + text
                    else:
                        transcript = text
                    prev_partial = ""
                    # Print final result on a new line
                    print("\n[FINAL] ", transcript)
            else:
                # partial result arrived
                p = json.loads(rec.PartialResult()).get('partial', '').strip()
                # only update output when partial changes (reduce flicker)
                if p != prev_partial:
                    prev_partial = p
                    # merged view = confirmed transcript + current partial
                    if transcript and p:
                        merged = (transcript + " " + p).strip()
                    elif p:
                        merged = p
                    else:
                        merged = transcript
                    # write carriage return to overwrite the same line
                    sys.stdout.write(("\r[PARTIAL] " + merged + " ")[:200])
                    sys.stdout.flush()

        # Get final result after processing all data
        final_result = json.loads(rec.FinalResult())
        text = final_result.get('text', '').strip()
        if text:
            if transcript:
                transcript = transcript + " " + text
            else:
                transcript = text
            print("\n[FINAL] ", transcript)

    except FileNotFoundError:
        print("Error: sample.wav not found. Please place a file named 'sample.wav' in the project folder.", file=sys.stderr)
    except Exception as e:
        print("\nError:", str(e))


if __name__ == "__main__":
    main()


Processing audio from file: sample.wav

Loading model from: vosk-model-small-en-us-0.15
[PARTIAL] hello how are we are testing this code 
[FINAL]  hello how are we are testing this code
